In [10]:
from dcase2017.task1.datamodule import AcousticScenesDatamodule as DM
from dcase2017.task1.models import BaselineModel as Model
from dcase2017.task1.baseline_experiment import BaselineExperiment as Exp
import torch
import yaml
from tqdm import tqdm
CONFIG_PATH = 'config/baseline.yaml'
CKPT_PATH = '/export/home/kienegger/speakerbeam_jnf/data/logs/dcase2017/ckpts/baseline/epoch=116-val_acc=0.65.ckpt'

In [11]:
with open(CONFIG_PATH, 'r') as file:
        config = yaml.safe_load(file)
config['network']['sample_rate'] = config['data']['sample_rate']
config['network']['n_label'] = config['experiment']['n_label']
dm = DM(**config['data'])
model = Model(
    **config['network'], 
)
exp = Exp.load_from_checkpoint(
    CKPT_PATH,
    model=model, 
    **config['experiment'], 
)

In [12]:
acc_score = []
for example in tqdm(dm.test_dataloader()):
    with torch.no_grad():
        # load data
        audio_data = example['audio_data'].to(device=exp.device)  # (BATCH, CHANNEL, TIME)
        target_label = example['class_label'].to(device=exp.device)  # (BATCH)
        
        # forward model
        unnormalized_logits = exp.model(audio_data)  # (BATCH, FRAMES', CLASS)

        # compute accuracy with majority voting
        est_label = torch.argmax(unnormalized_logits, dim=-1).squeeze(dim=1)  # (BATCH, FRAMES')
        est_label = torch.stack(
            [
                torch.bincount(e_label).argmax() for e_label in est_label
            ]
        )  # (BATCH)
        acc_score.append(
            exp.accuracy(est_label, target_label)
        )

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 540/540 [00:13<00:00, 41.46it/s]


In [16]:
print(
    CKPT_PATH.split('/')[-1] + f' with accuracy {torch.tensor(acc_score).mean().cpu().data.numpy()}'
)

epoch=116-val_acc=0.65.ckpt with accuracy 0.644444465637207
